## 0 · Environment Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import os
IN_COLAB = "COLAB_GPU" in os.environ or Path("/content").exists()

if IN_COLAB:
    import kagglehub
    path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
    os.environ["DATA_DIR"] = str(Path(path) / "chest_xray")
    print(f"Dataset downloaded to: {os.environ['DATA_DIR']}")

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

from src.config import (
    CLASS_NAMES,
    DATA_DIR,
    DEVICE,
    LEARNING_RATE,
    NUM_EPOCHS,
    RESULTS_DIR,
)
from src.datasets import compute_class_weights, get_dataloaders, get_transforms
from src.evaluate import collect_predictions, compute_metrics, plot_confusion_matrix
from src.models import CustomCNN, build_densenet121
from src.train import run_training

print(f"Device : {DEVICE}")
print(f"Data   : {DATA_DIR}")
print(f"Exists : {DATA_DIR.exists()}")

## 1 · Load Data & Class Weights

In [ ]:
loaders = get_dataloaders(DATA_DIR)

for split, loader in loaders.items():
    print(f"  {split:<6}  {len(loader.dataset):>5} images  {len(loader):>4} batches")

train_ds = loaders["train"].dataset
class_weights = compute_class_weights(train_ds).to(DEVICE)

print(f"\nClass weights: {dict(zip(CLASS_NAMES, class_weights.tolist()))}")

## 2 · Smoke Test (1 Batch, Both Models)

In [ ]:
batch_imgs, batch_lbls = next(iter(loaders["train"]))
batch_imgs = batch_imgs.to(DEVICE)

# Custom CNN
cnn_test = CustomCNN().to(DEVICE)
out_cnn = cnn_test(batch_imgs)
assert out_cnn.shape == (batch_imgs.size(0), 2), f"CustomCNN shape: {out_cnn.shape}"
del cnn_test

# DenseNet121
dn_test = build_densenet121().to(DEVICE)
out_dn = dn_test(batch_imgs)
assert out_dn.shape == (batch_imgs.size(0), 2), f"DenseNet shape: {out_dn.shape}"
del dn_test

torch.cuda.empty_cache() if torch.cuda.is_available() else None
print(f"Smoke test PASSED — both models output shape ({batch_imgs.size(0)}, 2)")

---
## 3 · Train Custom CNN

In [ ]:
model_cnn = CustomCNN().to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model_cnn.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

cnn_history = run_training(
    model_cnn,
    loaders,
    criterion,
    optimizer,
    scheduler=scheduler,
    num_epochs=NUM_EPOCHS,
    model_name="custom_cnn",
)

### Training Curves — Custom CNN

In [ ]:
def plot_training_curves(history: dict, title: str) -> None:
    """Plot loss and accuracy curves for train and val."""
    epochs = range(1, len(history["train_history"]) + 1)
    train_loss = [e["loss"] for e in history["train_history"]]
    train_acc  = [e["accuracy"] for e in history["train_history"]]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(epochs, train_loss, label="Train")
    ax2.plot(epochs, train_acc,  label="Train")

    if history["val_history"]:
        val_loss = [e["loss"] for e in history["val_history"]]
        val_acc  = [e["accuracy"] for e in history["val_history"]]
        ax1.plot(epochs, val_loss, label="Val")
        ax2.plot(epochs, val_acc,  label="Val")

    ax1.set_title(f"{title} — Loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend()

    ax2.set_title(f"{title} — Accuracy")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.legend()

    fig.tight_layout()
    fig.savefig(RESULTS_DIR / f"{title.lower().replace(' ', '_')}_curves.png", dpi=150)
    plt.show()

plot_training_curves(cnn_history, "Custom CNN")

### Evaluate Custom CNN on Test Set

In [ ]:
y_true_cnn, y_pred_cnn = collect_predictions(model_cnn, loaders["test"], DEVICE)
metrics_cnn = compute_metrics(y_true_cnn, y_pred_cnn)

plot_confusion_matrix(
    y_true_cnn, y_pred_cnn,
    save_path=RESULTS_DIR / "cm_custom_cnn.png",
)

report = metrics_cnn["report"]
print("Custom CNN — Test Set Results")
print(f"  Accuracy  : {report['accuracy']:.4f}")
print(f"  Recall    : {report['PNEUMONIA']['recall']:.4f}")
print(f"  Precision : {report['PNEUMONIA']['precision']:.4f}")
print(f"  F1-Score  : {report['PNEUMONIA']['f1-score']:.4f}")

---
## 4 · Train DenseNet121 (Transfer Learning)

In [ ]:
model_dn = build_densenet121(pretrained=True).to(DEVICE)

# Phase 1: freeze backbone, train only the classifier head
for param in model_dn.features.parameters():
    param.requires_grad = False

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer_ph1 = optim.Adam(model_dn.classifier.parameters(), lr=1e-3)

print("Phase 1 — Frozen backbone (5 epochs)")
dn_history_ph1 = run_training(
    model_dn,
    loaders,
    criterion,
    optimizer_ph1,
    num_epochs=5,
    model_name="densenet121_ph1",
)

In [ ]:
# Phase 2: unfreeze all layers, fine-tune with lower LR
for param in model_dn.features.parameters():
    param.requires_grad = True

optimizer_ph2 = optim.Adam(model_dn.parameters(), lr=1e-4)
scheduler_ph2 = optim.lr_scheduler.StepLR(optimizer_ph2, step_size=5, gamma=0.1)

print("Phase 2 — Full fine-tune (15 epochs)")
dn_history_ph2 = run_training(
    model_dn,
    loaders,
    criterion,
    optimizer_ph2,
    scheduler=scheduler_ph2,
    num_epochs=15,
    model_name="densenet121",
)

### Training Curves — DenseNet121

In [ ]:
# Merge phase 1 + phase 2 histories for a single plot
dn_history_combined = {
    "train_history": dn_history_ph1["train_history"] + dn_history_ph2["train_history"],
    "val_history": dn_history_ph1["val_history"] + dn_history_ph2["val_history"],
    "best_val_accuracy": max(dn_history_ph1["best_val_accuracy"], dn_history_ph2["best_val_accuracy"]),
}

plot_training_curves(dn_history_combined, "DenseNet121")

### Evaluate DenseNet121 on Test Set

In [ ]:
y_true_dn, y_pred_dn = collect_predictions(model_dn, loaders["test"], DEVICE)
metrics_dn = compute_metrics(y_true_dn, y_pred_dn)

plot_confusion_matrix(
    y_true_dn, y_pred_dn,
    save_path=RESULTS_DIR / "cm_densenet121.png",
)

report_dn = metrics_dn["report"]
print("DenseNet121 — Test Set Results")
print(f"  Accuracy  : {report_dn['accuracy']:.4f}")
print(f"  Recall    : {report_dn['PNEUMONIA']['recall']:.4f}")
print(f"  Precision : {report_dn['PNEUMONIA']['precision']:.4f}")
print(f"  F1-Score  : {report_dn['PNEUMONIA']['f1-score']:.4f}")

---
## 5 · Side-by-Side Comparison

In [ ]:
r_cnn = metrics_cnn["report"]
r_dn  = metrics_dn["report"]

header = f"{'Metric':<12} {'Custom CNN':>12} {'DenseNet121':>12}"
sep    = "-" * len(header)

rows = [
    ("Accuracy",  r_cnn["accuracy"],                 r_dn["accuracy"]),
    ("Recall",    r_cnn["PNEUMONIA"]["recall"],       r_dn["PNEUMONIA"]["recall"]),
    ("Precision", r_cnn["PNEUMONIA"]["precision"],    r_dn["PNEUMONIA"]["precision"]),
    ("F1-Score",  r_cnn["PNEUMONIA"]["f1-score"],     r_dn["PNEUMONIA"]["f1-score"]),
]

print(header)
print(sep)
for name, val_cnn, val_dn in rows:
    best = " <--" if val_dn > val_cnn else (" <--" if val_cnn > val_dn else "")
    marker_cnn = " <--" if val_cnn > val_dn else ""
    marker_dn  = " <--" if val_dn > val_cnn else ""
    print(f"{name:<12} {val_cnn:>11.4f}{marker_cnn}  {val_dn:>11.4f}{marker_dn}")

print(f"\nBest val acc — CNN: {cnn_history['best_val_accuracy']:.4f}  "
      f"DenseNet: {dn_history_combined['best_val_accuracy']:.4f}")